# 27. Unsupervised Learning: Gaussian Mixture Models (GMM)

## Algorithm Category
**Type**: Unsupervised Learning - Clustering & Density Estimation  
**Complexity**: Medium-High  
**Use Case**: Probabilistic clustering using mixture of Gaussian distributions

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand Gaussian Mixture Models and probabilistic clustering
- Implement GMM using Expectation-Maximization (EM) algorithm
- Understand soft clustering vs hard clustering
- Use information criteria (AIC, BIC) to select number of components
- Visualize probability distributions and cluster assignments
- Apply GMM to real-world problems

## Historical Context

GMM with EM algorithm was popularized in the 1970s-1980s:
- Dempster, A.P., et al. (1977): "Maximum likelihood from incomplete data via the EM algorithm"
- One of the most important algorithms in statistics and machine learning
- Foundation for many probabilistic models

**Key Papers/References:**
- Dempster, A.P., et al. (1977). "Maximum likelihood from incomplete data via the EM algorithm"
- McLachlan, G. & Peel, D. (2000). "Finite Mixture Models"

## When to Use Gaussian Mixture Models

GMM is appropriate when:
- You need soft/probabilistic clustering
- Data can be modeled as mixture of Gaussians
- You want to estimate probability distributions
- Clusters may overlap
- You need uncertainty estimates for cluster assignments
- Working with continuous data

## Theory & Mechanics

### Mathematical Foundation

GMM models data as a mixture of K Gaussian distributions:

**Probability Density Function:**
$$p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

Where:
- $\pi_k$: Mixing coefficient (weight) for component $k$, $\sum_{k=1}^{K} \pi_k = 1$
- $\mu_k$: Mean of component $k$
- $\Sigma_k$: Covariance matrix of component $k$
- $\mathcal{N}(x | \mu_k, \Sigma_k)$: Multivariate Gaussian distribution

**Expectation-Maximization (EM) Algorithm:**

**E-Step (Expectation):**
$$\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x_i | \mu_j, \Sigma_j)}$$

**M-Step (Maximization):**
$$\mu_k = \frac{\sum_{i=1}^{N} \gamma_{ik} x_i}{\sum_{i=1}^{N} \gamma_{ik}}$$

$$\Sigma_k = \frac{\sum_{i=1}^{N} \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T}{\sum_{i=1}^{N} \gamma_{ik}}$$

$$\pi_k = \frac{1}{N} \sum_{i=1}^{N} \gamma_{ik}$$

### How It Works

1. **Initialize**: Randomly initialize parameters ($\mu_k$, $\Sigma_k$, $\pi_k$)
2. **E-Step**: Calculate responsibility (posterior probability) of each point belonging to each component
3. **M-Step**: Update parameters using weighted maximum likelihood
4. **Repeat**: Steps 2-3 until convergence

### Key Hyperparameters

- **n_components**: Number of mixture components (clusters)
- **covariance_type**: Type of covariance matrix
  - 'full': Each component has its own general covariance matrix
  - 'tied': All components share the same covariance matrix
  - 'diag': Each component has its own diagonal covariance matrix
  - 'spherical': Each component has its own single variance
- **max_iter**: Maximum iterations for EM algorithm
- **tol**: Convergence threshold

### Advantages

- Provides soft clustering (probabilistic assignments)
- Can model overlapping clusters
- Handles elliptical clusters (not just spherical)
- Estimates probability distributions
- Uses information criteria (AIC, BIC) for model selection

### Limitations

- Assumes Gaussian distribution (may not fit all data)
- Sensitive to initialization (local optima)
- Can be slow for large datasets
- Requires specifying number of components
- May struggle with non-Gaussian clusters


## Implementation

Let's implement Gaussian Mixture Models.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Import our helper functions
from src.models.unsupervised import evaluate_clustering

print("Libraries imported successfully!")


In [ ]:
# Generate synthetic dataset
X, y_true = make_blobs(n_samples=300, centers=3, n_features=2, 
                       random_state=42, cluster_std=0.60)

print(f"Dataset Shape: {X.shape}")
print(f"True number of clusters: {len(np.unique(y_true))}")

# Apply Gaussian Mixture Model
gmm = GaussianMixture(n_components=3, random_state=42, covariance_type='full')
gmm.fit(X)
y_pred = gmm.predict(X)
y_proba = gmm.predict_proba(X)

print(f"\nGMM Results:")
print(f"  Number of components: {gmm.n_components}")
print(f"  Covariance type: {gmm.covariance_type}")
print(f"  Converged: {gmm.converged_}")
print(f"  Number of iterations: {gmm.n_iter_}")

# Visualize
plt.figure(figsize=(15, 5))

# True clusters
plt.subplot(1, 3, 1)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
plt.title('True Clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)

# Hard clustering (predict)
plt.subplot(1, 3, 2)
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', marker='x', 
           s=200, linewidths=3, label='Means')
plt.title('GMM Hard Clustering')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

# Soft clustering (probabilities)
plt.subplot(1, 3, 3)
# Color by probability of belonging to cluster 0
plt.scatter(X[:, 0], X[:, 1], c=y_proba[:, 0], cmap='Reds', s=50, alpha=0.7)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='blue', marker='x', 
           s=200, linewidths=3, label='Means')
plt.title('GMM Soft Clustering\n(Probability of Cluster 0)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar(label='Probability')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Model Selection: AIC and BIC

Let's use information criteria to select the optimal number of components.


In [ ]:
# Test different numbers of components
n_components_range = range(1, 11)
aic_scores = []
bic_scores = []
silhouette_scores = []

for n_comp in n_components_range:
    gmm_test = GaussianMixture(n_components=n_comp, random_state=42, covariance_type='full')
    gmm_test.fit(X)
    aic_scores.append(gmm_test.aic(X))
    bic_scores.append(gmm_test.bic(X))
    
    if n_comp > 1:
        labels = gmm_test.predict(X)
        sil_score = silhouette_score(X, labels)
        silhouette_scores.append(sil_score)
    else:
        silhouette_scores.append(-1)

# Plot information criteria
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(n_components_range, aic_scores, 'o-', label='AIC', markersize=6)
axes[0].plot(n_components_range, bic_scores, 's-', label='BIC', markersize=6)
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Score')
axes[0].set_title('AIC and BIC for Model Selection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=3, color='r', linestyle='--', label='True k=3')
axes[0].legend()

axes[1].plot(n_components_range, silhouette_scores, '^-', color='green', markersize=6)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=3, color='r', linestyle='--', label='True k=3')
axes[1].legend()

plt.tight_layout()
plt.show()

optimal_aic = n_components_range[np.argmin(aic_scores)]
optimal_bic = n_components_range[np.argmin(bic_scores)]
optimal_sil = n_components_range[np.argmax(silhouette_scores)]

print(f"Optimal number of components:")
print(f"  AIC: {optimal_aic}")
print(f"  BIC: {optimal_bic}")
print(f"  Silhouette: {optimal_sil}")
print(f"  True: 3")


## Covariance Types

Let's compare different covariance types.


In [ ]:
# Compare different covariance types
covariance_types = ['full', 'tied', 'diag', 'spherical']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, cov_type in enumerate(covariance_types):
    gmm_cov = GaussianMixture(n_components=3, random_state=42, covariance_type=cov_type)
    gmm_cov.fit(X)
    labels_cov = gmm_cov.predict(X)
    
    axes[idx].scatter(X[:, 0], X[:, 1], c=labels_cov, cmap='viridis', s=50, alpha=0.7)
    axes[idx].scatter(gmm_cov.means_[:, 0], gmm_cov.means_[:, 1], c='red', marker='x', 
                     s=200, linewidths=3)
    axes[idx].set_title(f'{cov_type.capitalize()}\nAIC: {gmm_cov.aic(X):.1f}')
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nComparison of covariance types:")
for cov_type in covariance_types:
    gmm_cov = GaussianMixture(n_components=3, random_state=42, covariance_type=cov_type)
    gmm_cov.fit(X)
    print(f"  {cov_type}: AIC = {gmm_cov.aic(X):.1f}, BIC = {gmm_cov.bic(X):.1f}")


## Validation & Testing

Let's validate the model and compare with K-Means.


In [ ]:
# Evaluate clustering
evaluation = evaluate_clustering(X, y_pred, algorithm='GMM')
print("Clustering Evaluation:")
print(f"  Silhouette Score: {evaluation['silhouette_score']:.3f}")
print(f"  Number of clusters: {evaluation['n_clusters']}")

# Compare with K-Means
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
y_kmeans = kmeans.fit_predict(X)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', marker='x', 
           s=200, linewidths=3, label='GMM Means')
plt.title('GMM (Soft Clustering)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, cmap='viridis', s=50, alpha=0.7)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='K-Means Centroids')
plt.title('K-Means (Hard Clustering)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nComparison:")
print(f"  GMM AIC: {gmm.aic(X):.1f}")
print(f"  GMM BIC: {gmm.bic(X):.1f}")
print(f"  GMM provides probability estimates, K-Means does not")

# Assertions
assert gmm.converged_, "GMM should converge"
assert evaluation['silhouette_score'] > 0, "Silhouette score should be positive"
print("\n✓ Validation checks passed")


## Real-World Application

Let's apply GMM to the Iris dataset.


In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Scale features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Apply GMM
gmm_iris = GaussianMixture(n_components=3, random_state=42, covariance_type='full')
gmm_iris.fit(X_iris_scaled)
y_iris_pred = gmm_iris.predict(X_iris_scaled)
y_iris_proba = gmm_iris.predict_proba(X_iris_scaled)

# Evaluate
sil_score_iris = silhouette_score(X_iris_scaled, y_iris_pred)
print("Iris Dataset Clustering:")
print(f"  Number of components: {gmm_iris.n_components}")
print(f"  Silhouette Score: {sil_score_iris:.3f}")
print(f"  AIC: {gmm_iris.aic(X_iris_scaled):.1f}")
print(f"  BIC: {gmm_iris.bic(X_iris_scaled):.1f}")

# Visualize (using first two features)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('True Labels')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris_pred, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('GMM Clustering (k=3)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show probability distribution for a sample
sample_idx = 0
print(f"\nSample {sample_idx} probability distribution:")
print(f"  True label: {iris.target_names[y_iris[sample_idx]]}")
print(f"  Predicted: {y_iris_pred[sample_idx]}")
print(f"  Probabilities: {y_iris_proba[sample_idx]}")
for i, prob in enumerate(y_iris_proba[sample_idx]):
    print(f"    Cluster {i}: {prob:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **GMM Basics**
   - Probabilistic clustering model
   - Models data as mixture of Gaussian distributions
   - Provides soft clustering (probability assignments)
   - Uses EM algorithm for parameter estimation

2. **Expectation-Maximization (EM)**
   - **E-Step**: Calculate responsibilities (posterior probabilities)
   - **M-Step**: Update parameters using weighted MLE
   - Iterates until convergence

3. **Model Selection**
   - **AIC (Akaike Information Criterion)**: Penalizes complexity less
   - **BIC (Bayesian Information Criterion)**: Stronger penalty for complexity
   - Lower is better for both
   - Use to select optimal number of components

4. **Covariance Types**
   - **Full**: Each component has own general covariance (most flexible)
   - **Tied**: All components share same covariance
   - **Diag**: Diagonal covariance matrices
   - **Spherical**: Single variance per component (like K-Means)

### When to Use Gaussian Mixture Models

✅ **Good for:**
- Need soft/probabilistic clustering
- Overlapping clusters
- Elliptical clusters (not just spherical)
- When you need probability estimates
- Density estimation
- Continuous data

❌ **Not ideal for:**
- Non-Gaussian data distributions
- Very large datasets (can be slow)
- When hard clustering is sufficient (use K-Means)
- High-dimensional data (curse of dimensionality)
- Discrete/categorical data

### Next Steps

- Compare with **K-Means** for hard clustering
- Use **Bayesian GMM** for automatic component selection
- Apply to **anomaly detection** (low probability points)
- Use for **density estimation** tasks
- Explore **variational inference** for faster training
